In [1]:
import os

import papermill as pm
from IPython.display import Image, display

from jaxcmr.helpers import find_project_root

# LohnasKahana2014 Model_Fitting

Fit multiple CMR variants to the Lohnas & Kahana 2014 free recall dataset.

**Paradigm:** Free Recall
**Reference:** @lohnas2014retrieved

## Dataset Configuration

In [2]:
trial_query = "data['list_type'] > 0"
mixed_trial_query = "data['list_type'] == 2"
control_trial_query = "data['list_type'] == 1"

base_params = {
    "redo_fits": False,
    "redo_sims": False,
    "redo_figures": False,
    "handle_elis": False,
    "filter_repeated_recalls": False,
    "base_run_tag": "rerun",
    "experiment_count": 200,
    "max_subjects": 0,
    "base_data_tag": "Lohnas2025",
    "data_tag": "Lohnas2025",
    "data_path": "data/Lohnas2025.h5",
    "trial_query": trial_query,
    "target_directory": "projects/repfr/results/",
    "component_paths": {
        "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
        "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
        "context_create_fn": "jaxcmr.components.context.init",
        "termination_policy_create_fn": "jaxcmr.components.termination.NoStopTermination",
    },
    "sim_alg_path": "jaxcmr.simulation.simulate_study_free_recall_and_forced_stop",
    "loss_fn_path": "jaxcmr.loss.transform_sequence_likelihood.ExcludeTerminationLikelihoodFnGenerator",
    "fit_alg_path": "jaxcmr.fitting.ScipyDE",
    "seed": 0,
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 1,
    "comparison_analysis_configs": [
        {"target": "jaxcmr.analyses.spc.plot_spc", "figure_suffix": "spc"},
        {"target": "jaxcmr.analyses.crp.plot_crp", "figure_suffix": "crp"},
        {"target": "jaxcmr.analyses.pnr.plot_pnr", "figure_suffix": "pnr"},
    ],
    "single_analysis_configs": [],
    "template_render_configs": [
        {
            "template_path": "templates/repcrp.ipynb",
            "analysis_suffix": "repcrp",
            "params": {
                "control_shuffles": 1,
                "mixed_trial_query": mixed_trial_query,
                "control_trial_query": control_trial_query,
                "ylim": [0.05, 0.32],
            },
        },
        {
            "template_path": "templates/backrepcrp.ipynb",
            "analysis_suffix": "backrepcrp",
            "params": {
                "control_shuffles": 1,
                "mixed_trial_query": mixed_trial_query,
                "control_trial_query": control_trial_query,
                "ylim": [0.05, 0.32],
            },
        },
        {
            "template_path": "templates/repneighborcrp.ipynb",
            "analysis_suffix": "repneighborcrp",
            "params": {
                "control_shuffles": 1,
                "mixed_trial_query": mixed_trial_query,
                "control_trial_query": control_trial_query,
            },
        },
    ],
}


## Model Configurations

Core models for Chapter 3 repetition analysis:
1. **WeirdCMR** - Standard composite context model (mfc_choice_sensitivity free)
2. **WeirdPositionalCMR** - Instance-based context representation (mfc_choice_sensitivity fixed to 1.0)
3. **WeirdNoReinstateCMR** - CMR without context reinstatement
4. **WeirdCMRDistinctContexts** - CMR with separate context representations

In [3]:
varied_parameters = [
    # Baseline CMR
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "WeirdCMRNoStop",
        "make_factory_path": "jaxcmr.models.cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
            },
        },
    },
    # No Reinstate CMR
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "NoReinstateCMRNoStop",
        "make_factory_path": "jaxcmr.models.no_reinstate_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
            },
        },
    },
    # Distinct Contexts CMR
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "DistinctContextsCMRNoStop",
        "make_factory_path": "jaxcmr.models.distinct_contexts_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
            },
        },
    },
    # Positional CMR (base)
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "BasePositionalCMRNoStop",
        "make_factory_path": "jaxcmr.models.positional_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "mfc_sensitivity": 1.0,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
            },
        },
    },
    # Positional CMR (full)
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "FullPositionalCMRNoStop",
        "make_factory_path": "jaxcmr.models.positional_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "mfc_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
            },
        },
    },
    # MCF Reinf Positional CMR
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "McfReinfPositionalCMRNoStop",
        "make_factory_path": "jaxcmr.models.reinf_positional_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "mfc_sensitivity": 1.0,
                "mfc_first_pres_reinforcement": 0.0,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "mcf_first_pres_reinforcement": [
                    2.220446049250313e-16,
                    99.9999999999999998,
                ],
            },
        },
    },
    # MFC Reinf Positional CMR
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "MfcReinfPositionalCMRNoStop",
        "make_factory_path": "jaxcmr.models.reinf_positional_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "mfc_sensitivity": 1.0,
                "mcf_first_pres_reinforcement": 0.0,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "mfc_first_pres_reinforcement": [
                    2.220446049250313e-16,
                    99.9999999999999998,
                ],
            },
        },
    },
    # Full Reinf Positional CMR
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "FullReinfPositionalCMRNoStop",
        "make_factory_path": "jaxcmr.models.reinf_positional_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "mfc_sensitivity": 1.0,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "mcf_first_pres_reinforcement": [
                    2.220446049250313e-16,
                    99.9999999999999998,
                ],
                "mfc_first_pres_reinforcement": [
                    2.220446049250313e-16,
                    99.9999999999999998,
                ],
            },
        },
    },
    # Full Reinf Positional CMR (simple)
    {
        "redo_fits": False,
        "redo_sims": False,
        "redo_figures": False,
        "model_name": "SimpleFullReinfPositionalCMRNoStop",
        "make_factory_path": "jaxcmr.models.reinf_positional_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "mfc_sensitivity": 1.0,
                "mfc_first_pres_reinforcement": 0.0,
                "mcf_first_pres_reinforcement": 0.0,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "first_pres_reinforcement": [
                    2.220446049250313e-16,
                    99.9999999999999998,
                ],
            },
        },
    },
]

## Run Model Fitting

In [ ]:
for partial_params in varied_parameters:
    params = base_params.copy() | {
        k: v for k, v in partial_params.items() if k != "enabled"
    }

    data_tag = params["data_tag"]
    model_name = params["model_name"]
    base_run_tag = params["base_run_tag"]
    best_of = params["best_of"]
    fit_tag = f"{base_run_tag}_best_of_{best_of}"
    max_subjects = params["max_subjects"]
    if max_subjects:
        fit_tag += f"_nsubs_{max_subjects}"

    output_path = (
        f"{find_project_root()}/projects/repfr/notebooks/rendered/"
        f"fitting_{data_tag}_{model_name}_{fit_tag}.ipynb"
    )
    print(output_path)

    pm.execute_notebook(
        f"{find_project_root()}/projects/repfr/notebooks/templates/repetition_fitting.ipynb",
        output_path,
        parameters=params,
        progress_bar=True,
    )

    for analysis_cfg in params["single_analysis_configs"]:
        figure_str = (
            f"{data_tag}_{model_name}_{fit_tag}_{analysis_cfg['figure_suffix']}.png"
        )
        figure_path = os.path.join(
            f"{find_project_root()}/projects/repfr/results/figures/fitting/{figure_str}"
        )
        print(f"![]({figure_path})")
        display(Image(filename=figure_path, width=600))

    for analysis_cfg in params["comparison_analysis_configs"]:
        figure_str = (
            f"{data_tag}_{model_name}_{fit_tag}_{analysis_cfg['figure_suffix']}.png"
        )
        figure_path = os.path.join(
            f"{find_project_root()}/projects/repfr/results/figures/fitting/{figure_str}"
        )
        print(f"![]({figure_path})")
        display(Image(filename=figure_path, width=600))


Unable to parse line 16 'control_trial_query = "data['list_type'] == 1"'.


/Users/jordangunn/jaxcmr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/jordangunn/jaxcmr/projects/repfr/notebooks/rendered/fitting_Lohnas2025_WeirdCMRNoStop_rerun_best_of_1.ipynb


Executing:  40%|████      | 6/15 [00:02<00:02,  3.16cell/s]ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/jordangunn/jaxcmr/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/jordangunn/jaxcmr/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 341, in dispatch_control
    await self.process_control(msg)
  File "/Users/jordangunn/jaxcmr/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 347, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jordangunn/jaxcmr/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 993, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/User